In [1]:
import gradio as gr
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
import cv2

# Store uploaded images and labels
data = []
labels = []
label_names = []

def upload_image(img, label):
    if label not in label_names:
        label_names.append(label)
    img_resized = cv2.resize(img, (64, 64))
    data.append(img_resized)
    labels.append(label_names.index(label))
    return f"Image added to class: {label}"

def train_model():
    if len(set(labels)) < 2:
        return "At least 2 classes are required to train."
    
    X = np.array(data)
    y = np.array(labels)

    X = X / 255.0
    y = tf.keras.utils.to_categorical(y, num_classes=len(set(labels)))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    model = models.Sequential([
        layers.Conv2D(16, (3,3), activation='relu', input_shape=(64,64,3)),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(32, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(len(set(labels)), activation='softmax')
    ])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

    model.save("teachable_model.h5")
    return "Training complete! Model saved as teachable_model.h5"

def predict_image(img):
    model = tf.keras.models.load_model("teachable_model.h5")
    img_resized = cv2.resize(img, (64, 64)) / 255.0
    img_resized = np.expand_dims(img_resized, axis=0)
    pred = model.predict(img_resized)
    class_idx = np.argmax(pred)
    return f"Predicted class: {label_names[class_idx]}"

# Gradio UI
upload_interface = gr.Interface(
    fn=upload_image,
    inputs=[gr.Image(type="numpy"), gr.Textbox(label="Class Label")],
    outputs="text",
    title="Upload Training Image"
)

train_interface = gr.Interface(
    fn=train_model,
    inputs=[],
    outputs="text",
    title="Train Model"
)

predict_interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="numpy"),
    outputs="text",
    title="Predict"
)

gr.TabbedInterface(
    [upload_interface, train_interface, predict_interface],
    ["Upload", "Train", "Predict"]
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


C:\Users\LENOVO\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 1.0000 - loss: 0.4335 - val_accuracy: 0.0000e+00 - val_loss: 3.6086
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 1.0000 - loss: 0.0162 - val_accuracy: 0.0000e+00 - val_loss: 5.7012
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step - accuracy: 1.0000 - loss: 0.0015 - val_accuracy: 0.0000e+00 - val_loss: 7.5842
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 468ms/step - accuracy: 1.0000 - loss: 1.6783e-04 - val_accuracy: 0.0000e+00 - val_loss: 9.2940
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - accuracy: 1.0000 - loss: 2.3365e-05 - val_accuracy: 0.0000e+00 - val_loss: 10.8883


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
